# OpenClip on Kaggle
Settings → **Accelerator: GPU T4 x2** and **Internet: On**.
Secrets (Add-ons → Secrets): `LLM_BASE_URL` (your gateway's **tunnel** URL, e.g. `https://xxx.trycloudflare.com` — 127.0.0.1 is your PC, Kaggle can't reach it), `LLM_API_KEY`; optional `GEMINI_API_KEY`, `YOUTUBE_COOKIES` (Netscape format).
Run all cells; the last one prints a public URL for the dashboard.

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
def secret(name):
    try: return UserSecretsClient().get_secret(name)
    except Exception: return None

os.chdir('/kaggle/working')
if not os.path.exists('OpenClip'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/sanupyadav/OpenClip.git'], check=True)
os.chdir('/kaggle/working/OpenClip')
!git log --oneline -1

In [ ]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq ffmpeg fontconfig fonts-liberation fonts-noto-color-emoji > /dev/null
# Node 20 for the dashboard
if ! node -v 2>/dev/null | grep -q '^v2[0-9]'; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null
  apt-get install -y -qq nodejs > /dev/null
fi
# Bundled fonts for libass (same as the Dockerfile)
mkdir -p /usr/local/share/fonts/openshorts
cp fonts/*.ttf /usr/local/share/fonts/openshorts/
cp fonts/openshorts-fontmap.conf /etc/fonts/conf.d/60-openshorts.conf
fc-cache -f > /dev/null
# cloudflared quick tunnel
[ -x /usr/local/bin/cloudflared ] || curl -fsSL -o /usr/local/bin/cloudflared \
  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
node -v; ffmpeg -version | head -1

In [ ]:
%%bash
set -e
# Keep Kaggle's CUDA torch: the pinned torch wheel may target a CUDA the T4 driver lacks.
grep -vE '^(torch|torchvision)==' requirements.txt > /tmp/req.txt
pip install -q -r /tmp/req.txt
pip install -q "nvidia-cublas-cu12<13" "nvidia-cudnn-cu12>=9,<10" onnx-asr onnxruntime-gpu
pip install -q --upgrade --pre "yt-dlp[default]"
cd dashboard && npm install --no-audit --no-fund --loglevel=error

In [ ]:
import re, pathlib
# Let the tunnel hostname through Vite's host check.
cfg = pathlib.Path('dashboard/vite.config.js')
cfg.write_text(re.sub(r'allowedHosts:\s*\[[^\]]*\]', 'allowedHosts: true', cfg.read_text()))

site = subprocess.check_output(['python', '-c', 'import site; print(site.getsitepackages()[0])'], text=True).strip()
env = dict(os.environ,
    PYTHONUNBUFFERED='1',
    WHISPER_MODEL='large-v3-turbo', WHISPER_DEVICE='cuda', WHISPER_COMPUTE='float16',
    FFMPEG_ENCODER='auto', MAX_CONCURRENT_JOBS='1',
    LD_LIBRARY_PATH=f'{site}/nvidia/cublas/lib:{site}/nvidia/cudnn/lib:' + os.environ.get('LD_LIBRARY_PATH', ''),
)
for k in ('GEMINI_API_KEY', 'YOUTUBE_COOKIES'):
    if (v := secret(k)): env[k] = v

# Moment picker through your OpenAI-compatible gateway, Gemini model.
PUBLIC_URL = secret('LLM_BASE_URL')  # tunnel URL of the gateway
if PUBLIC_URL:
    env['LLM_BASE_URL'] = PUBLIC_URL.rstrip('/').removesuffix('/v1') + '/v1'
    env['LLM_API_KEY'] = secret('LLM_API_KEY') or ''
    env['LLM_MODEL'] = 'google/gemini-3.8-flash'
    env['LLM_SCORE_BATCH'] = '8'

os.makedirs('output', exist_ok=True)
backend = subprocess.Popen(['uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
                           env=env, stdout=open('backend.log', 'w'), stderr=subprocess.STDOUT)
frontend = subprocess.Popen(['npm', 'run', 'dev', '--', '--host', '0.0.0.0', '--port', '5173'], cwd='dashboard',
                            env=dict(env, VITE_PROXY_TARGET='http://localhost:8000'),
                            stdout=open('frontend.log', 'w'), stderr=subprocess.STDOUT)
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:5173'],
                          stdout=open('tunnel.log', 'w'), stderr=subprocess.STDOUT)

import time
for _ in range(60):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
    if m: break
print('Dashboard:', m.group(0) if m else 'not ready, check tunnel.log')

In [ ]:
# Logs (re-run any time). Clips land in /kaggle/working/OpenClip/output.
!tail -n 30 backend.log; tail -n 10 frontend.log

In [ ]:
# Keep the session alive while you use the dashboard (interrupt to stop).
import time
while backend.poll() is None: time.sleep(60)
print('backend exited:', backend.returncode)